# Testing YOLO On Shooting Drills

Testing the chosen YOLO pose model (`yolo11l-pose`) on footage of shooting drills from several different angles.

## 0. Setup

In [ ]:
import os

%pip install ultralytics==8.4.26 -q

if not os.path.exists("football-kick-tracker"):
    !git clone https://github.com/WillEdgington/football-kick-tracker.git
else:
    !git -C football-kick-tracker pull

%pip install -e football-kick-tracker --force-reinstall -q

import sys

sys.path.insert(0, '/content/football-kick-tracker')

import IPython

IPython.display.clear_output()
print("Setup complete. Runtime restarting...")
exit()

In [ ]:
from pathlib import Path

from google.colab import drive
from ultralytics import YOLO

from pose.annotate import annotateVideos
from pose.config import (
    ANNOTATEDPOSEVIDEOSDIR,
    CONFTHRESHOLD,
    LOWERCOLOURS,
    RAWTRAININGVIDEOSDIR,
)
from pose.constants import LOWERKEYPOINTS, LOWERSKELETONPAIRS

drive.mount('/content/drive')
driveRepoDir = Path("/content/drive/MyDrive/football-kick-tracker")

## 1. Define video paths

Get paths for shooting drills from an established directory (prefix "shooting-" is used on file names).

In [ ]:
videoDir: Path = driveRepoDir / RAWTRAININGVIDEOSDIR
videoPaths = sorted(
    p for suf in ("*.mp4", "*.mov")
    for p in videoDir.glob(suf)
    if p.stem.startswith("shooting-")
)

assert len(videoPaths) > 0, f"No shooting videos found in {videoDir}"
print(f"Found {len(videoPaths)} videos:")
for p in videoPaths:
    print(f"  {p.name}")

## 2. Load YOLO models

Collect chosen model

In [ ]:
modelName = "yolo11l-pose"
try:
    model = YOLO(f"{modelName}.pt")
    print(f"{modelName} loaded")
except Exception as e:
    print(f"{modelName} not available: {e}")

Check that models are on GPU

In [ ]:
!nvidia-smi

print(model.device)

## 3. Annotate Shooting Videos

In [ ]:
annotateVideosDir: Path = driveRepoDir / ANNOTATEDPOSEVIDEOSDIR

annotatedVideoPaths = annotateVideos(
    model=model,
    name=modelName,
    sourcePaths=videoPaths,
    outputDir=annotateVideosDir,
    keypointIndexes=LOWERKEYPOINTS,
    skeletonPairs=LOWERSKELETONPAIRS,
    colours=LOWERCOLOURS,
    confThreshold=CONFTHRESHOLD,
    logging=True,
    overwrite=False
)

## 4. Observations

**Experiment Structure:**

- Used `yolo11l-pose` model (See `YOLO_candidate_comparison.ipynb` notebook for reason why this is the current model of choice)
- Annotated several high quality videos of one shooting drill session (outside, sunny, ~one coach, one player, one ball in action) from several different camera angles.
- Did not evaluate keypoint confidence and detection rate like in `YOLO_candidate_comparison.ipynb` due to the single-player setting on the footage (we would be rewarding confident hallucinations if we did).
- Decided that visual evaluation of annotated clips was the most accurate and time efficient method to see the model's performance (manual labelling will be needed at a later date in the project).

Camera angles observed (*{horizontal to subject}-{height}-{distance}*):

*NOTE: low height is ~hip level, mid and high is drone level*
- behind-high-mid
- behind-mid-far
- behind-mid-mid
- behind-low-mid
- behind-low-close
- side-low-far

**Notes:**
- Model works best at hip level and at a close to mid range distance (the closer and more direct the angle, the better the model worked).
- Model failed at far distance (this was similar to what was seen from the rondo clips tested in `YOLO_candidate_comparison.ipynb`).
- Model also failed mostly at drone footage (at behind-high-mid we also had quite heavy hallucinations due to shadows).
- Model performed much better at behind-mid-far than it did at behind-mid-mid (this shows that the more the subject's shape is foreshortened, the harder it is for the model).
- There was the occasional hallucination on the football mannequins (not enough for it to be a big concern).
- Overall the model was a lot less jittery than I thought it would be. When it worked it worked well, when it failed it failed heavily.

**Decision:**
- Cements that future experimentation on non-YOLO alternatives and possible fine-tuning is needed to make pose estimation robust for the task at hand.
- `yolo11l-pose` remains sufficient to progress with for now. However, I do doubt its use in production.